**Purpose:** complete story of the pipeline in one notebook —
architecture, data flow, quality results, and key findings.

In [0]:
from pyspark.sql import functions as F

CATALOG_NAME = "iot_pipeline"
SCHEMA_NAME  = "sensors"

BRONZE_TABLE   = f"{CATALOG_NAME}.{SCHEMA_NAME}.bronze_sensors"
SILVER_TABLE   = f"{CATALOG_NAME}.{SCHEMA_NAME}.silver_sensors"
GOLD_HOURLY    = f"{CATALOG_NAME}.{SCHEMA_NAME}.gold_hourly_stats"
GOLD_DAILY     = f"{CATALOG_NAME}.{SCHEMA_NAME}.gold_daily_health"
GOLD_ANOMALIES = f"{CATALOG_NAME}.{SCHEMA_NAME}.gold_anomalies"
QUALITY_TABLE  = f"{CATALOG_NAME}.{SCHEMA_NAME}.quality_report"

print("All tables referenced. This notebook is read-only.")

---
## Section 1 — Pipeline health at a glance

The first thing any stakeholder wants to see:
did the pipeline run successfully and how much data made it through?

In [0]:
# We use Delta metadata here (DESCRIBE DETAIL) rather than
# COUNT(*) for Bronze and Silver — fast, no full scan needed.
# For Gold tables we use COUNT(*) since they're small
# aggregated tables where a scan is cheap.
# ============================================================

bronze_count = spark.sql(f"SELECT COUNT(*) AS n FROM {BRONZE_TABLE}").collect()[0]["n"]
silver_count = spark.sql(f"SELECT COUNT(*) AS n FROM {SILVER_TABLE}").collect()[0]["n"]
hourly_count = spark.sql(f"SELECT COUNT(*) AS n FROM {GOLD_HOURLY}").collect()[0]["n"]
daily_count  = spark.sql(f"SELECT COUNT(*) AS n FROM {GOLD_DAILY}").collect()[0]["n"]
anomaly_count= spark.sql(f"SELECT COUNT(*) AS n FROM {GOLD_ANOMALIES}").collect()[0]["n"]

retained_pct = round(silver_count / bronze_count * 100, 1)
removed_count= bronze_count - silver_count

print("=" * 55)
print("  PIPELINE ROW FLOW")
print("=" * 55)
print(f"  Bronze (raw)          : {bronze_count:>7,} rows")
print(f"  Removed in cleaning   : {removed_count:>7,} rows  ({100-retained_pct:.1f}%)")
print(f"  Silver (clean)        : {silver_count:>7,} rows  ({retained_pct}% retained)")
print(f"  Gold hourly stats     : {hourly_count:>7,} rows  (aggregated)")
print(f"  Gold daily health     : {daily_count:>7,} rows  (aggregated)")
print(f"  Gold anomalies        : {anomaly_count:>7,} rows  flagged")
print("=" * 55)

In [0]:
latest_run = spark.sql(f"""
    SELECT MAX(run_id) AS latest FROM {QUALITY_TABLE}
""").collect()[0]["latest"]

print(f"Showing results for run: {latest_run}\n")

print("=== Quality scorecard ===")
display(spark.sql(f"""
    SELECT
        layer,
        COUNT(*)                                         AS total_checks,
        SUM(CASE WHEN status = 'PASS' THEN 1 ELSE 0 END) AS passed,
        SUM(CASE WHEN status = 'WARN' THEN 1 ELSE 0 END) AS warned,
        SUM(CASE WHEN status = 'FAIL' THEN 1 ELSE 0 END) AS failed,
        CONCAT(
            ROUND(
                SUM(CASE WHEN status = 'PASS' THEN 1 ELSE 0 END)
                / COUNT(*) * 100
            , 0), '%'
        )                                                AS pass_rate
    FROM {QUALITY_TABLE}
    WHERE run_id = '{latest_run}'
    GROUP BY layer
    ORDER BY layer
"""))

---
## Section 2 — What the data looks like

Seven days of sensor readings across 5 factory machines.
Three sensor types: temperature, pressure, vibration.

In [0]:
# ============================================================
# We show data coverage in two dimensions:
# 1. Time — do we have readings across all 7 days?
# 2. Machines — is every machine represented?
#
# Gaps in either dimension would indicate a data collection
# problem — a machine went offline or a sensor failed.
# In a real pipeline this view would be checked every day.
# ============================================================

print("=== Daily reading counts per machine ===")
print("(This confirms we have consistent data across all machines and days)\n")

display(spark.sql(f"""
    SELECT
        TO_DATE(timestamp)  AS date,
        machine_id,
        COUNT(*)            AS readings,
        COUNT(DISTINCT sensor_type) AS sensor_types_active
    FROM {SILVER_TABLE}
    GROUP BY TO_DATE(timestamp), machine_id
    ORDER BY date, machine_id
"""))

In [0]:
# ============================================================
# This shows the statistical profile of clean Silver data.
# We compare mean and stddev across machines for each sensor
# type to check if any machine behaves differently from others.
#
# In a real factory, if MACHINE_03's average temperature is
# consistently 15 degrees higher than the other machines,
# that's worth investigating even if it's within range.
# This kind of cross-machine comparison is something a
# data engineer adds value with — not just "is the data clean"
# but "does the data tell us something interesting".
# ============================================================

print("=== Sensor value profile per machine (Silver data) ===")
display(spark.sql(f"""
    SELECT
        machine_id,
        sensor_type,
        COUNT(*)                    AS readings,
        ROUND(AVG(value), 3)        AS mean,
        ROUND(STDDEV(value), 3)     AS stddev,
        ROUND(MIN(value), 3)        AS min_val,
        ROUND(MAX(value), 3)        AS max_val
    FROM {SILVER_TABLE}
    GROUP BY machine_id, sensor_type
    ORDER BY sensor_type, machine_id
"""))

---
## Section 3 — Machine health findings

The Gold daily health table answers one business question:
**which machines were operating normally and which need attention?**

In [0]:
# ============================================================
# This is the most business-relevant output of the pipeline.
# We summarise across all 7 days to give an overall picture,
# then show the worst single day per machine so nothing is hidden.
# ============================================================

print("=== Overall machine health (averaged across all days) ===")
display(spark.sql(f"""
    SELECT
        machine_id,
        ROUND(AVG(health_score_pct), 1)     AS avg_health_score,
        MIN(health_score_pct)               AS worst_day_score,
        MAX(health_score_pct)               AS best_day_score,
        -- Most severe status seen across all days
        CASE
            WHEN MIN(health_score_pct) < 80  THEN 'CRITICAL'
            WHEN MIN(health_score_pct) < 95  THEN 'WARNING'
            ELSE 'NORMAL'
        END                                  AS overall_status,
        SUM(total_readings)                  AS total_readings
    FROM {GOLD_DAILY}
    GROUP BY machine_id
    ORDER BY avg_health_score ASC
"""))

In [0]:
# ============================================================
# The day-by-day view shows trends — is a machine getting
# worse over time or was it a one-off bad day?
# A machine that goes NORMAL → NORMAL → WARNING → CRITICAL
# over 4 days needs urgent attention. A machine that had
# one WARNING day and is otherwise NORMAL is lower priority.
#
# This kind of trend visibility is exactly what makes a
# pipeline valuable to the business — not just data storage,
# but decision support.
# ============================================================

print("=== Day-by-day machine health ===")
display(spark.sql(f"""
    SELECT
        date,
        machine_id,
        health_score_pct,
        machine_status,
        total_readings,
        ROUND(avg_temperature, 2) AS avg_temp,
        ROUND(avg_pressure, 2)    AS avg_pressure,
        ROUND(avg_vibration, 4)   AS avg_vibration
    FROM {GOLD_DAILY}
    ORDER BY date, machine_id
"""))

---
## Section 4 — Anomaly findings

Anomalies are sensor readings that deviated more than
2 standard deviations from that machine's rolling baseline.

These are the readings that would trigger an alert
in a production monitoring system.

In [0]:
# ============================================================
# We present anomalies in two views:
# 1. Summary — which machine/sensor combination has the
#    highest anomaly rate? That's where to focus attention.
# 2. Top anomalies — the most extreme individual readings,
#    sorted by z-score. The highest z-score = furthest from
#    normal = most urgent to investigate.
# ============================================================

print("=== Anomaly rate per machine and sensor ===")
display(spark.sql(f"""
    SELECT
        machine_id,
        sensor_type,
        COUNT(*)                    AS anomaly_count,
        ROUND(MAX(z_score), 2)      AS max_z_score,
        ROUND(AVG(z_score), 2)      AS avg_z_score,
        ROUND(MIN(value), 4)        AS min_anomalous_value,
        ROUND(MAX(value), 4)        AS max_anomalous_value
    FROM {GOLD_ANOMALIES}
    GROUP BY machine_id, sensor_type
    ORDER BY anomaly_count DESC
"""))

print("\n=== Top 10 most extreme anomalies (highest z-score) ===")
display(spark.sql(f"""
    SELECT
        machine_id,
        sensor_type,
        ROUND(value, 4)         AS value,
        unit,
        ROUND(rolling_avg, 4)   AS rolling_avg,
        ROUND(z_score, 2)       AS z_score,
        timestamp
    FROM {GOLD_ANOMALIES}
    ORDER BY z_score DESC
    LIMIT 10
"""))

In [0]:
# ============================================================
# Anomalies distributed over time tells us if problems are
# random (expected — sensor noise) or clustered on specific
# days (concerning — something happened to the machine).
#
# If 80% of anomalies for MACHINE_02 happened on day 4,
# that is a strong signal that something went wrong on that
# day — worth correlating with maintenance logs.
# ============================================================

print("=== Anomalies per day per machine ===")
display(spark.sql(f"""
    SELECT
        TO_DATE(timestamp)  AS date,
        machine_id,
        sensor_type,
        COUNT(*)            AS anomaly_count,
        ROUND(AVG(z_score), 2) AS avg_z_score
    FROM {GOLD_ANOMALIES}
    GROUP BY TO_DATE(timestamp), machine_id, sensor_type
    ORDER BY date, machine_id, sensor_type
"""))

---
## Section 5 — Key findings & recommendations

This section summarises the pipeline findings that can be 
presented to a non-technical stakeholder.

In [0]:
# ============================================================
# We compute the key numbers for the findings summary
# dynamically — not hardcoded — so this cell stays accurate
# even if you re-run the pipeline with different data.
# ============================================================

# Worst performing machine
worst_machine = spark.sql(f"""
    SELECT machine_id, ROUND(AVG(health_score_pct), 1) AS avg_score
    FROM {GOLD_DAILY}
    GROUP BY machine_id
    ORDER BY avg_score ASC
    LIMIT 1
""").collect()[0]

# Most anomalous sensor type
most_anomalous_sensor = spark.sql(f"""
    SELECT sensor_type, COUNT(*) AS n
    FROM {GOLD_ANOMALIES}
    GROUP BY sensor_type
    ORDER BY n DESC
    LIMIT 1
""").collect()[0]

# Highest single z-score
max_anomaly = spark.sql(f"""
    SELECT machine_id, sensor_type, ROUND(z_score, 2) AS z_score
    FROM {GOLD_ANOMALIES}
    ORDER BY z_score DESC
    LIMIT 1
""").collect()[0]

# Quality pass rate
quality_pass = spark.sql(f"""
    SELECT
        ROUND(
            SUM(CASE WHEN status = 'PASS' THEN 1 ELSE 0 END)
            / COUNT(*) * 100
        , 0) AS pass_rate
    FROM {QUALITY_TABLE}
    WHERE run_id = '{latest_run}'
""").collect()[0]["pass_rate"]

print("=" * 55)
print("  KEY FINDINGS")
print("=" * 55)
print(f"  Data period         : 7 days, 5 machines")
print(f"  Total raw readings  : {bronze_count:,}")
print(f"  Clean readings      : {silver_count:,} ({retained_pct}% of raw)")
print(f"  Pipeline quality    : {quality_pass}% checks passed")
print(f"  Anomalies detected  : {anomaly_count:,} readings flagged")
print(f"")
print(f"  Machine needing attention : {worst_machine['machine_id']}")
print(f"  Average health score      : {worst_machine['avg_score']}%")
print(f"")
print(f"  Most anomalous sensor     : {most_anomalous_sensor['sensor_type']}")
print(f"  Anomaly count             : {most_anomalous_sensor['n']:,}")
print(f"")
print(f"  Most extreme reading      : {max_anomaly['machine_id']} / {max_anomaly['sensor_type']}")
print(f"  Z-score                   : {max_anomaly['z_score']} std deviations from normal")
print("=" * 55)